# Randomized benchmarking

This notebook shows a compact randomized benchmarking workflow: prepare a
good single-qubit gate set, run standard RB, and then estimate specific
gate errors with interleaved RB.

## 1. Create an `Experiment`

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [ ]:
Q0 = exp.qubit_labels[0]

print("target qubit:", Q0)

## 2. Connect to the setup

Connect before you run the prerequisite calibrations.

In [ ]:
exp.connect()

## 3. Check the waveform

A waveform check is a quick sanity check before you spend time on benchmarking.

In [ ]:
waveform_result = exp.check_waveform()

## 4. Measure a baseline Rabi oscillation

Use the baseline Rabi result to confirm that the drive amplitude is in a usable range.

In [ ]:
rabi_result = exp.obtain_rabi_params()

## 5. Calibrate the DRAG half-pi pulse

Run the DRAG half-pi calibration first so the primitive gate set is in good shape.

In [ ]:
drag_hpi_result = exp.calibrate_drag_hpi_pulse()

## 6. Repeat the DRAG half-pi pulse

Use a repeated-sequence check to confirm that the calibrated half-pi pulse accumulates as expected.

In [ ]:
repeat_drag_hpi = exp.repeat_sequence(exp.drag_hpi_pulse)

## 7. Calibrate the DRAG pi pulse

Next, calibrate the DRAG pi pulse used in the benchmarking gate set.

In [ ]:
drag_pi_result = exp.calibrate_drag_pi_pulse()

## 8. Repeat the DRAG pi pulse

Check the calibrated pi pulse with the same repeated-sequence workflow.

In [ ]:
repeat_drag_pi = exp.repeat_sequence(exp.drag_pi_pulse)

## 9. Save the calibration note

Save the calibrated pulses before you move on to the benchmarking measurements.

In [ ]:
exp.calib_note.save()

## 10. Inspect the calibrated pulses

Plot the DRAG half-pi and pi pulses for the target qubit so you can inspect the cached waveforms directly.

In [ ]:
drag_hpi = exp.drag_hpi_pulse[Q0]
drag_hpi.plot()

drag_pi = exp.drag_pi_pulse[Q0]
drag_pi.plot()

## 11. Run standard randomized benchmarking

Use the calibrated single-qubit gate set to estimate the average Clifford error.

In [ ]:
rb_result = exp.randomized_benchmarking(
    Q0,
    n_cliffords_range=np.arange(0, 1001, 100),
    n_trials=30,
    save_image=True,
)

## 12. Run interleaved RB for `X90`

Interleave `X90` with the RB sequence to estimate the contribution of that gate.

In [ ]:
irb_x90 = exp.interleaved_randomized_benchmarking(
    Q0,
    interleaved_clifford="X90",
    n_cliffords_range=range(0, 1001, 100),
    n_trials=30,
    save_image=True,
)

## 13. Run interleaved RB for `X180`

Repeat the interleaved analysis for `X180`.

In [ ]:
irb_x180 = exp.interleaved_randomized_benchmarking(
    Q0,
    interleaved_clifford="X180",
    n_cliffords_range=range(0, 1001, 100),
    n_trials=30,
    save_image=True,
)